# Treino experimental da voz da Lia (Piper Plus / VITS)

**Notebook preparado para execução manual no Google Colab.** Ao executar o passo de upload, os arquivos de voz serão transferidos do seu computador para o runtime Google. Você autorizou esse envio nesta conversa. O notebook não monta Drive, não publica no Hugging Face e não envia áudio para terceiros além do ambiente Colab escolhido. O runtime é temporário; interrompa e apague a sessão ao terminar.

## Condição de licença

Candidato técnico: `ayousanz/piper-plus-base` (Piper Plus multilingual, inclui português). O model card publica dados/receita, mas não declara claramente a licença dos pesos. O código do Piper Plus se declara MIT, o que **não licencia os pesos**. Este notebook bloqueia o treino por padrão; libere-o somente para prova privada/local se você aceitar a ressalva. Não publique nem redistribua o checkpoint/base/fine-tune sem confirmar os direitos.

Não use áudio de outra pessoa sem autorização. Revise transcrições manualmente. Este fluxo não faz ASR. Mantenha originais privados fora do Git.

**Entrada:** um ZIP preparado com `metadata.csv` no formato `id|texto` e arquivos `.wav` mono correspondentes. Os exemplos devem ser falas naturais, sem cortes de fonemas. Separe frases inéditas para avaliação antes de treinar.

In [ ]:
# Controles explícitos. Nenhum upload ocorre nesta célula.
PRIVATE_EVALUATION_ONLY = True
ALLOW_UNRESOLVED_BASE_WEIGHTS_FOR_PRIVATE_TEST = False  # só altere se aceitar a ressalva de licença

if not PRIVATE_EVALUATION_ONLY:
    raise RuntimeError('Este notebook não publica modelos; use apenas em avaliação privada.')
if not ALLOW_UNRESOLVED_BASE_WEIGHTS_FOR_PRIVATE_TEST:
    raise RuntimeError('Treino bloqueado: confirme a licença dos pesos ou aceite explicitamente apenas teste privado.')
print('Autorização registrada nesta sessão; nenhum áudio foi enviado ainda.')

## 1. Ambiente do treino

Instalações ocorrem apenas dentro do Colab, não no Windows nem no repositório. Código fixado no commit consultado em 2026-09-30. Confirme no runtime que há GPU e espaço suficientes. Se a receita tiver mudado, pare e confira o guia upstream.

In [ ]:
import pathlib, subprocess
REPO = pathlib.Path('/content/piper-plus')
PIN = '03f22b895d6b87d3733223c0844cb2ef6569937b'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/ayutaz/piper-plus.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout',PIN], check=True)
subprocess.run(['python','-m','pip','install','uv'], check=True)
subprocess.run(['uv','pip','install','--system','.[train]'], cwd=REPO, check=True)
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'GPU NVIDIA não reportada')

## 2. Upload explícito do dataset

Execute somente quando estiver pronto para enviar o ZIP do computador ao runtime Colab. O ZIP e as transcrições ficam temporariamente no runtime. Não inclua arquivos sem autorização de uso.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil
uploaded = files.upload()  # abre seletor local; envia somente o ZIP escolhido
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Envie exatamente um ZIP com metadata.csv e WAVs mono.')
DATA_ROOT = Path('/content/lia_voice_data')
shutil.rmtree(DATA_ROOT, ignore_errors=True)
DATA_ROOT.mkdir(parents=True)
with zipfile.ZipFile(zip_names[0]) as zf:
    root = DATA_ROOT.resolve()
    for info in zf.infolist():
        target = (DATA_ROOT / info.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError('Caminho inseguro dentro do ZIP; extração bloqueada.')
    zf.extractall(DATA_ROOT)
print('Extraído no armazenamento temporário:', DATA_ROOT)

## 3. Validar estrutura e áudio

`metadata.csv` usa pipe (`id|texto`), sem cabeçalho. Cada identificador precisa corresponder a `id.wav`. O pré-processamento Piper converte para 22.050 Hz. Revise transcrições, nomes, números, clipping, ruído e cortes antes do treino.

In [ ]:
import soundfile as sf
DATA_ROOT = Path('/content/lia_voice_data')
meta_files = list(DATA_ROOT.rglob('metadata.csv'))
if len(meta_files) != 1:
    raise ValueError(f'Esperado um metadata.csv; encontrados {len(meta_files)}')
META = meta_files[0]
AUDIO_DIR = META.parent
rows, missing, durations, rates = [], [], [], set()
for line_no, line in enumerate(META.read_text(encoding='utf-8').splitlines(), 1):
    if not line.strip() or line.lstrip().startswith('#'): continue
    parts = line.split('|', 1)
    if len(parts) != 2 or not all(part.strip() for part in parts):
        raise ValueError(f'Linha {line_no} inválida; esperado id|texto')
    key, text = parts[0].strip(), parts[1].strip()
    wav = AUDIO_DIR / (key + '.wav')
    if not wav.is_file():
        missing.append((line_no, key)); continue
    info = sf.info(wav)
    if info.channels != 1 or info.frames == 0:
        raise ValueError(f'{wav.name}: esperado WAV mono não vazio')
    rows.append((key, text)); durations.append(info.duration); rates.add(info.samplerate)
if missing: raise FileNotFoundError(f'Áudios ausentes: {missing[:10]}')
if not rows: raise ValueError('Dataset vazio')
print(f'{len(rows)} frases; {sum(durations)/60:.1f} min; duração {min(durations):.2f}-{max(durations):.2f} s')
print('Sample rates encontrados:', sorted(rates))
print('Validação estrutural concluída; faça revisão auditiva e transcrição antes de avançar.')

## 4. Localizar base, revisar licença e preparar

Baixe manualmente o checkpoint e `config.json` do model card oficial, confira revisão, hash e licença. Não use o Faber apenas porque já existe no PC: é outro checkpoint, com licença do peso não esclarecida. O candidate Piper Plus base também tem licença de peso não indicada, por isso a execução fica restrita a avaliação privada enquanto isso não for esclarecido.

In [ ]:
BASE_CHECKPOINT = '/content/piper_plus_base/model.ckpt'
BASE_CONFIG = '/content/piper_plus_base/config.json'
DATASET_DIR = '/content/lia_dataset'
OUT_DIR = '/content/lia_voice_experiment'
if not Path(BASE_CHECKPOINT).is_file() or not Path(BASE_CONFIG).is_file():
    raise FileNotFoundError('Coloque o checkpoint e config obtidos do model card oficial após revisar licença/revisão/hash.')
preprocess_cmd = [
    'python','-m','piper_train.preprocess',
    '--input-dir',str(AUDIO_DIR),'--output-dir',DATASET_DIR,
    '--language','pt','--dataset-format','ljspeech','--sample-rate','22050',
    '--single-speaker','--phoneme-type','multilingual'
]
print('Pré-processamento proposto:', ' '.join(preprocess_cmd))
RUN_PREPROCESS = False
if RUN_PREPROCESS:
    subprocess.run(preprocess_cmd, cwd=REPO, check=True)
else:
    print('Bloqueado por padrão. Após revisar caminhos/dados, mude RUN_PREPROCESS=True.')

## 5. Treino piloto

A receita abaixo é uma adaptação dos parâmetros publicados para Piper Plus multilingual. Começa com 20 épocas para testar viabilidade; não é promessa de qualidade final. Revise o dataset de validação, GPU/VRAM, licença e configuração antes de executar. **Treino também está bloqueado por padrão.**

In [ ]:
train_cmd = [
    'python','-m','piper_train','--dataset-dir',DATASET_DIR,
    '--prosody-dim','16','--accelerator','gpu','--devices','1',
    '--precision','32-true','--max_epochs','20','--batch-size','4',
    '--samples-per-speaker','4','--checkpoint-epochs','5',
    '--base_lr','2e-5','--disable_auto_lr_scaling','--ema-decay','0.9995',
    '--max-phoneme-ids','400','--no-wavlm',
    '--resume-from-multispeaker-checkpoint',BASE_CHECKPOINT,
    '--default_root_dir',OUT_DIR
]
print('Comando piloto:', ' '.join(train_cmd))
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(train_cmd, cwd=REPO, check=True)
else:
    print('Treino bloqueado por padrão; defina RUN_TRAINING=True após revisão consciente.')

## 6. Exportar, avaliar e apagar

Confirme no guia do commit fixado o comando de export ONNX adequado ao checkpoint produzido; não assuma compatibilidade pelo nome do comando. Avalie frases inéditas, compare com o baseline e escute antes de integrar. Não publique em Hub.

Ao terminar, baixe somente artefatos que decidiu conservar. Interrompa e apague o runtime Colab; o notebook não monta Drive e não faz uploads automáticos de pesos. Cópias que você baixar para PC/Drive persistem até serem removidas manualmente.